# Airline On-Time Performance and Turnaround Reliability Control Tower

Company inspiration: **British Airways**

Use this notebook as a working template. Replace prompts with your own analysis and decisions.

## 1. Business framing
- Who is the executive audience?
- What decision should this analysis improve?
- What are the core KPIs and business constraints?

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
base = Path.cwd().parents[1]
raw = base / 'B_Dataset' / 'raw'
meta = base / 'B_Dataset' / 'metadata'


## 2. Load and profile the data

In [11]:
#load files
df_flights = pd.read_csv("flight_operations.csv")
df_reliability = pd.read_csv("airport_daily_reliability.csv")
df_weather = pd.read_csv("airport_weather_daily.csv")

In [13]:
#clean column names
for df in [df_flights, df_reliability, df_weather]:
    df.columns = df.columns.str.strip().str.lower()

In [14]:
#fixed date columns
df_flights["flight_date"] = pd.to_datetime(df_flights["flight_date"])
df_reliability["flight_date"] = pd.to_datetime(df_reliability["flight_date"])
df_weather["date"] = pd.to_datetime(df_weather["date"])

In [16]:
#merge weather in to df_flights
df_flights = df_flights.merge(
    df_weather,
    left_on=["flight_date", "origin"],
    right_on=["date", "airport"],
    how="left"
)

In [18]:
 #merg reliability in to flights
df_final = df_flights.merge(
    df_reliability,
    on=["flight_date", "origin"],
    how="left"
)

In [ ]:
# now each fligt include ,fligt deatail, delay info,weather conditions and Airport performance
#now we can answer
#does weather caouse delay?
#do certain airports perfform worse?
#does inbound delay affect departure?
#which flights are high risk?

## 3. Data cleaning and transformation
Document each assumption and create reproducible transformation logic.

In [19]:
#Handle null values
df_final = df_final.fillna(0)

In [20]:
#save final data
df_final.to_csv("final_control_tower_dataset.csv", index=False)

In [ ]:
#After merging again further data cleaning had been done 
#check data in merged data set
print(df_final.shape)
print(df_final.isnull().sum())

(75525, 31)
flight_id                   0
flight_date                 0
origin                      0
destination                 0
tail_number                 0
scheduled_departure_ts      0
scheduled_arrival_ts        0
actual_departure_ts         0
actual_arrival_ts           0
departure_delay_min         0
arrival_delay_min           0
turnaround_duration_min     0
inbound_delay_min           0
airport_congestion_index    0
maintenance_event_flag      0
crew_late_flag              0
aircraft_type               0
route_profile               0
d15_flag                    0
d0_flag                     0
primary_delay_code          0
date                        0
airport                     0
visibility_km               0
wind_kts                    0
rain_mm                     0
fog_flag                    0
storm_flag                  0
dep_flights                 0
avg_dep_delay               0
d15_rate                    0
dtype: int64


In [22]:
#Above results show that 75525 raws , 31 columns and 0 missing values
#so I confirmed cleaning is solid and merged worked correctly and data set is ready for further analysis.
#But there is date and airport which came from weather merge and are duplicate of flight_date and origin
#to fix that
df_final = df_final.drop(columns=["date", "airport"], errors="ignore")

In [23]:
#very data types
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 75525 entries, 0 to 75524
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   flight_id                 75525 non-null  str           
 1   flight_date               75525 non-null  datetime64[us]
 2   origin                    75525 non-null  str           
 3   destination               75525 non-null  str           
 4   tail_number               75525 non-null  str           
 5   scheduled_departure_ts    75525 non-null  str           
 6   scheduled_arrival_ts      75525 non-null  str           
 7   actual_departure_ts       75525 non-null  str           
 8   actual_arrival_ts         75525 non-null  str           
 9   departure_delay_min       75525 non-null  float64       
 10  arrival_delay_min         75525 non-null  float64       
 11  turnaround_duration_min   75525 non-null  float64       
 12  inbound_delay_min         755

In [24]:

#As we can see some "str" like scheduled_departure_ts and actual_depparture_ts ect we need to convert time columns to extract hours so it will be easy to calculate delay properly,extract hours of the day and can analyse peak time
time_cols =[
    "scheduled_departure_ts",
    "actual_departure_ts",
    "scheduled_arrival_ts",
    "actual_arrival_ts"
]

for col in time_cols:
    df_final[col] = pd.to_datetime(df_final[col], errors="coerce")

In [25]:
#create time in to hours
df_final["departure_hour"] = df_final["scheduled_departure_ts"].dt.hour

In [34]:
df_final[["scheduled_departure_ts", "departure_hour"]].head()

,scheduled_departure_ts,departure_hour
0,2025-01-01 11:53:00,11
1,2025-01-01 14:20:00,14
2,2025-01-01 08:06:00,8
3,2025-01-01 11:57:00,11
4,2025-01-01 05:53:00,5


In [27]:
#Creating Business KPI columns
df_final["delay_status"] = df_final["departure_delay_min"].apply(
    lambda x: "Delayed" if x > 15 else "On Time"
)

In [28]:
#Analysing risk
def risk_level(row):
    if row["inbound_delay_min"] > 15 or row["turnaround_duration_min"] < 60:
        return "High Risk"
    elif row["departure_delay_min"] > 0:
        return "Medium Risk"
    else:
        return "Low Risk"

df_final["risk_level"] = df_final.apply(risk_level, axis=1)

In [29]:
#turnround risk
df_final["turnaround_risk"] = df_final["turnaround_duration_min"].apply(
    lambda x: "Tight" if x < 60 else "Safe"
)

In [30]:
#final data set checked
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 75525 entries, 0 to 75524
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   flight_id                 75525 non-null  str           
 1   flight_date               75525 non-null  datetime64[us]
 2   origin                    75525 non-null  str           
 3   destination               75525 non-null  str           
 4   tail_number               75525 non-null  str           
 5   scheduled_departure_ts    75525 non-null  datetime64[us]
 6   scheduled_arrival_ts      75525 non-null  datetime64[us]
 7   actual_departure_ts       75525 non-null  datetime64[ns]
 8   actual_arrival_ts         75525 non-null  datetime64[ns]
 9   departure_delay_min       75525 non-null  float64       
 10  arrival_delay_min         75525 non-null  float64       
 11  turnaround_duration_min   75525 non-null  float64       
 12  inbound_delay_min         755

In [32]:
_# saving final data set
df_final.to_csv("final_control_tower_dataset.csv", index=False)


## 4. Core analysis / modeling
Use this section for KPI analysis, modeling, segmentation, or optimization.

In [44]:
#Create time_of_day column
df_final["departure_hour"].head()

0    11
1    14
2     8
3    11
4     5
Name: departure_hour, dtype: int32

In [45]:
# Airport delay analysis
df_final.groupby("origin")["departure_delay_min"].mean().sort_values(ascending=False)


origin
LHR    22.685690
LGW    19.692674
MAN    18.023976
Name: departure_delay_min, dtype: float64

In [52]:
#creating time of the display
def time_of_day(hour):
    if hour < 6:
        return "Early Morning"
    elif hour < 12:
        return "Morning"
    elif hour < 18:
        return "Afternoon"
    else:
        return "Evening"

df_final["time_of_day"] = df_final["departure_hour"].apply(time_of_day)

In [48]:
#Time of day Analysis
df_final.groupby("time_of_day")["departure_delay_min"].mean()

time_of_day
Afternoon        21.611767
Early Morning    21.542752
Evening          19.882033
Morning          21.550446
Name: departure_delay_min, dtype: float64

In [41]:
#Inbound delay impact by time
df_final.groupby("time_of_day")["inbound_delay_min"].mean()

time_of_day
Afternoon        10.107707
Early Morning     9.440667
Evening           9.475787
Morning           9.792135
Name: inbound_delay_min, dtype: float64

In [49]:
#Turnaround impact on delay
df_final.groupby("time_of_day")["turnaround_duration_min"].mean()

time_of_day
Afternoon        60.274137
Early Morning    59.674000
Evening          61.207803
Morning          60.136867
Name: turnaround_duration_min, dtype: float64

In [50]:
#Weather impact on delay 
df_final.groupby("storm_flag")["departure_delay_min"].mean()



storm_flag
0.0    21.479742
1.0    20.914376
Name: departure_delay_min, dtype: float64

In [ ]:
#Creating Dashboard
1.Excecutive summary
2.Root cause Analysis
3.Action or Recommendations


#Page 1
1.open power BIDesktop
2.Click get data----> Text/ CSV
3. click load

Creating KPIs
go to Modeling------------->New measure
for this I renamed my table to "Flights" for better readability in power BI


Average Departure Delay = AVERAGE(Flights[departure_delay_min])


D15 Rate = DIVIDE(SUM(Flights[D15_Flag]), COUNT(Flights[Flight_ID]))



Delay Propagation Rate =
DIVIDE(
    COUNTROWS(
        FILTER(flights,
            flights[inbound_delay_min] > 15 &&
            flights[departure_delay_min] > 15
        )
    ),
    COUNT(flights[flight_id])
),


#Turnaround Overrun Rate =

DIVIDE(
    COUNTROWS(FILTER(flights, flights[turnaround_duration_min] > 60)),
    COUNT(flights[flight_id])
)




Added KPI cards
1 Average Depature Delay
2 D15 Rate
Delay Propagation Rate
3 Turnaround Overrun Rate

Added Chart to the dashboard

1 Air port performance (Airport Driving delay) 
Here I used bar chart and added origin in axis and departure_delay_min in value and sorted by departure_delay_min to show worst performing airports

2 Time of day impact on delay(delay pattern across the day)
Here I used a column chart and added time_of_day in axis and departure_delay_min in value to show which time of the day is more prone to delay

Delay status split(ontime vs delayed flights)
Here I used a pie chart and added delay_status in legend and count of flight_id in value to show how many flights are delayed vs on time


And also addred some filters to the dashboard like time of the day, airport and aircraft_type to make it more interactive and user can drill down to specific insights.




Page 2 dashboard
Root cause analysis- why are flights getting delayed?

1.Inbound Vs Departure Delay(Delay Propergation:Inbound vs departutre)
Here I used a scatter plot and added inbound_delay_min in x axis and departure_delay_min in y axis to show the relationship between inbound delay and departure delay
Here we can see that there is a positive correlation between inbound delay and departure delay, which indicates that flights that arrive late are more likely to depart late as well.

2.delay by time of the day(Delay build up across the day)
Here I used Column chart and added time_of_day in axis and departure_delay_min in value to show how delay builds up across the day

To calculate time of the day I created Hour Column and then created time of the day column using DAX formula
MOdeling------------>New column
DEpature Hour = HOUR(Flights[scheduled_departure_ts])

Then I created time of the day column using this formula

time_of_day = 
SWITCH(
    TRUE(),
    Flights[Departure Hour] < 6, "Early Morning",
    Flights[Departure Hour] < 12, "Morning",
    Flights[Departure Hour] < 18, "Afternoon",
    "Evening"
)



3.Turnaround  time impact on delay(Turnaround pressure by time period)
Here I used a column chart and added time_of_day in axis and turnaround_duration_min in value to show how turnaround time varies across the day and how it might be impacting delay
  Here to Calculate Avrage Turnaround Duration I used DAX formula
  
Average Turnaround Duration = AVERAGE(Flights[turnaround_duration_min])

This KPI means that On average, how long aircraft take between arrival and next departure
 Turnaround duration reflects operational efficiency. Higher values indicate potential delays in aircraft readiness and ground operations


 4. Airport Congession impact on delay(Airport Congetion level)
 Here I used a bar chart and added origin in axis and Airport_congestion_index in value to show which airports are more congested and how it might be impacting delay

5Primary Couse of Delays
Here I used Donut Chart and primary_delay_Code in legend and count of flight_id in value to show the primary cause of delay and which one is more common  

As a final step I add filters to the dashboard like time of the day, airport and aircraft_type to make it more interactive and user can drill down to specific insights.
 
 
 Page 3 Dashboard
 
 OPerational Improvement Plan
 
1. Focus on high risk flights (Flights by risk Level)
 Here I used Bar Chart and added risk_level in axis and count of flight_id in value to show how many flights are high risk, medium risk and low risk and we can focus on high risk flights to improve overall performance
 
 
 2.Priority Airports(Airport reuiring Immidiate intervention)
 
 3. Time base intervention (When intervention matter most)
  Here I used Column chart and added time_of_Day in axis and Avg Departure Delay in Value to show when delay is more and we can focus on that time period for intervention
 
4 Next I add a text box and added some recommendations based on the analysis like
1. Reduce Inbound Delay Propagation

Prioritize early-day recovery
Improve aircraft rotation buffers

2. Focus on High-Risk Flights

Monitor flights with inbound delay > 15 min
Apply proactive scheduling adjustments

3. Optimize Airport Operations

Address congestion at high-delay airports
Improve ground handling efficiency

4. Adjust Afternoon Operations

Add buffer time during peak disruption window
Allocate additional operational resources


finaly Added KPI Cards
D15 Rate and Delay propergation rate to monitor the impact of interventions over time and track improvement in performance.




## 5. Business recommendations
Summarize the decision implications, trade-offs, and next steps.

In [ ]:
The executive dashboard provides a high-level view of operational reliability, highlighting delay levels, propagation effects, and key performance variations across airports and time periods


Page 2 dives into root cause analysis, exploring how factors like inbound delays, turnaround times, and airport congestion contribute to overall delay patterns, helping identify areas for improvement.
It clearly shows that delays are not random, They are driven by
1.Inbound delay propergations 
2.Thime of the day Accumulation
3. Airport Congessions
4.Operational Factors like turnaround time and weather

The route course Analysis shows that the delays are primarily driven by inbound delay propagation and Network Congessions. Delays accumulate throughout the day, impacting turnaround efficiancy and departure performance.
THis shows That flights are not delayed But delays propargate through the network due to inbound distruptions and operational constraints.
Therefore to mitigate delays, focus should be on improving inbound flight reliability, optimizing turnaround processes, and managing airport congestion effectively.

Based on the analysis, the primary opportunity lies in reducing delay propagation through improved inbound management, targeted intervention on high-risk flights, and operational adjustments during peak disruption periods
